# Phase 2 -- False Lead #1: The Sensor Ghost
### *Nine Meters of Silence*, Chapter 1: "Thermal Suicide"

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rjmachauthor/nine-meters-of-silence/blob/main/ch01/notebooks/phase2_sensor_ghost.ipynb)

**The claim in the book:** the IMU shows a sudden, massive phase shift during the flutter event. The team suspects the sensor is broken. Maya runs a Kalman Filter Residual Analysis -- if the sensor were broken, the residual should blow up. Instead, the residuals are flat. The sensor isn't lying. It's tracking reality perfectly.

**A realism note before we start:** classic gyro drift is a genuinely slow phenomenon -- real aviation-grade gyros drift on the order of 0.01-10 degrees *per hour*, far too slow to matter within a few seconds. So the 'broken sensor' scenario here models a **sudden bias fault** instead -- a real, documented, instant-onset IMU failure mode that's also a much closer physical match to the book's own description: 'a sudden, massive phase shift.' See `CALIBRATION.md` for the sourcing.

In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    !git clone -q https://github.com/rjmachauthor/nine-meters-of-silence.git
    sys.path.insert(0, 'nine-meters-of-silence/ch01')
else:
    sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from physics.sensor_ghost import generate_healthy_sensor_scenario, generate_broken_sensor_scenario

In [ ]:
healthy = generate_healthy_sensor_scenario()
broken = generate_broken_sensor_scenario()

print('Healthy sensor + real 20 Hz flutter -- mean NIS overall:', round(np.mean(healthy['nis']), 2), '(expected ~1.0)')
print('Broken sensor (sudden bias fault) -- mean NIS in final 2 seconds:', round(np.mean(broken['nis'][-2000:]), 2), '(expected: much higher)')

In [ ]:
# Animate: the raw signal on top, the residual test (NIS) below, for both scenarios side by side
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex='col')

axes[0,0].set_title('Healthy sensor, real flutter')
axes[0,1].set_title('Broken sensor (sudden bias fault)')
axes[0,0].set_ylabel('Roll angle (rad)')
axes[1,0].set_ylabel('NIS (residual test)')
axes[1,0].set_xlabel('Time (s)')
axes[1,1].set_xlabel('Time (s)')

for ax in axes[0]:
    ax.set_xlim(0, 10); ax.set_ylim(-0.3, 0.6)
for ax in axes[1]:
    ax.set_xlim(0, 10); ax.set_ylim(0, 15)
    ax.axhline(1.0, color='gray', linestyle=':', label='expected (NIS=1)')
    ax.legend(loc='upper left', fontsize=8)

sig_h, = axes[0,0].plot([], [], color='seagreen', lw=0.8)
sig_b, = axes[0,1].plot([], [], color='crimson', lw=0.8)
nis_h, = axes[1,0].plot([], [], color='seagreen', lw=1.2)
nis_b, = axes[1,1].plot([], [], color='crimson', lw=1.2)

n_frames = 150
frame_idx = np.linspace(0, len(healthy['time']) - 1, n_frames).astype(int)

def update(i):
    idx = frame_idx[i]
    sig_h.set_data(healthy['time'][:idx+1], healthy['measured'][:idx+1])
    sig_b.set_data(broken['time'][:idx+1], broken['measured'][:idx+1])
    nis_h.set_data(healthy['time'][:idx+1], healthy['nis'][:idx+1])
    nis_b.set_data(broken['time'][:idx+1], broken['nis'][:idx+1])
    return sig_h, sig_b, nis_h, nis_b

ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=50, blit=True)
plt.tight_layout()
plt.close(fig)
HTML(ani.to_jshtml())

Notice: the *signals* on top look similarly dramatic in both cases. The difference only shows up in the residual test on the bottom -- the healthy sensor's NIS hovers around the expected value of 1 throughout, even during the wild flutter oscillation, while the broken sensor's NIS jumps and stays elevated once the bias fault hits. That's the actual diagnostic Maya runs -- not 'does this look like a big problem', but 'is the disagreement between prediction and measurement statistically normal.'

## Try it yourself

In [ ]:
# --- Play with this ---
my_bias_magnitude = 0.2   # book scenario: 0.2 rad (~11.5 degrees). Try 0.0 (no fault) or 0.4 (worse fault)
# ------------------------

test_run = generate_broken_sensor_scenario(bias_magnitude_rad=my_bias_magnitude)
late_nis = np.mean(test_run['nis'][-1000:])
print(f"With bias fault of {my_bias_magnitude} rad: mean NIS in final second = {late_nis:.2f}")
print("(expected ~1.0 for a healthy sensor; well above that means the residual test would catch it)")